### middleware

In [ ]:
# --- Middleware: hooks around the agent loop without rewriting it (NOTES.md §7) ---
# SummarizationMiddleware auto-summarizes old messages to fit the context window —
# it's a before_model hook (runs before each LLM call). Conceptually it's the same
# work you'd otherwise wire as an extra node; middleware just keeps create_agent a
# one-liner. trigger = when to summarize, keep = how much recent history to retain.
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage


model=init_chat_model("ollama:qwen3:8b")
agent=create_agent(
    model=model,
    checkpointer=InMemorySaver(),   # memory across turns within a thread (NOTES.md §5)
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("messages", 3),  # summarize once history passes 3 messages
            keep=("messages", 1)      # keep the most recent message verbatim
        )
    ]
)
### message size
### token size
### fraction

### run with thread id — all turns share one conversation history (NOTES.md §5)
config={"configurable":{"thread_id":"test-1"}}
questions=[
    "what is 1 + 1?",
    "what is 2 + 1?",
    "what is 3 + 1?",
    "what is 4 + 1?",
    "what is 5 + 1?",
]

# Watch the message count: it doesn't grow unbounded because summarization collapses
# old turns once the trigger threshold is crossed.
for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages Count: {len(response["messages"])}")